# RealBasicVSR — Colab Master Setup

**Purpose:** Low-resolution video → 4× RealBasicVSR AI upscale → 1688×960 MP4.

This notebook is designed to be reused in a new Google Colab account/runtime.

**Known working environment**
- Python 3.10.x
- PyTorch 1.13.1+cu117
- torchvision 0.14.1
- mmcv-full 1.7.2
- mmedit 0.16.1
- openmim 0.3.9
- NumPy 1.26.4
- Tested successfully on Tesla T4

**Important:** GPU is required for the AI upscaling step. Final FFmpeg encoding does not require GPU.


## STEP 0 — Select a GPU runtime

In Colab choose **Runtime → Change runtime type → T4 GPU**. Then run the next cell.

In [ ]:
!nvidia-smi


## STEP 1 — Clone RealBasicVSR

In [ ]:
!git clone https://github.com/ckkelvinchan/RealBasicVSR.git /content/RealBasicVSR


## STEP 2 — Create the known-working Python environment

Do not upgrade these packages to newer major versions during this workflow.

In [ ]:
!python3 -m venv /content/rbvsr_legacy
!/content/rbvsr_legacy/bin/python -m pip install --upgrade "pip<24"
!/content/rbvsr_legacy/bin/pip install "numpy==1.26.4"
!/content/rbvsr_legacy/bin/pip install torch==1.13.1+cu117 torchvision==0.14.1 --extra-index-url https://download.pytorch.org/whl/cu117
!/content/rbvsr_legacy/bin/pip install openmim==0.3.9
!/content/rbvsr_legacy/bin/mim install "mmcv-full==1.7.2"
!/content/rbvsr_legacy/bin/pip install "mmedit==0.16.1"
!/content/rbvsr_legacy/bin/pip install opencv-python tqdm


## STEP 3 — Verify versions and CUDA

Expected: NumPy 1.26.4, Torch 1.13.1+cu117, torchvision 0.14.1, MMCV 1.7.2, MMEdit 0.16.1, and CUDA available = True.

In [ ]:
!/content/rbvsr_legacy/bin/python -c "import sys,torch,torchvision,numpy,mmcv,mmedit; print('Python:',sys.version); print('Torch:',torch.__version__); print('Torchvision:',torchvision.__version__); print('NumPy:',numpy.__version__); print('MMCV:',mmcv.__version__); print('MMEdit:',mmedit.__version__); print('CUDA available:',torch.cuda.is_available()); print('GPU:',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"


## STEP 4 — Check project files

In [ ]:
!ls -lh /content/RealBasicVSR/configs/realbasicvsr_x4.py
!ls -lh /content/RealBasicVSR/inference_realbasicvsr.py


## STEP 5 — RealBasicVSR checkpoint

Place `RealBasicVSR.pth` at:

`/content/RealBasicVSR/checkpoints/RealBasicVSR.pth`

The checkpoint was previously verified successfully and was about 201 MB.


In [ ]:
!mkdir -p /content/RealBasicVSR/checkpoints
!ls -lh /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
!/content/rbvsr_legacy/bin/python -c "import torch; p='/content/RealBasicVSR/checkpoints/RealBasicVSR.pth'; x=torch.load(p,map_location='cpu'); print(x.keys())"


## STEP 6 — Upload the input video

Use the upload button in the Colab Files panel, or run the next cell.

The example below expects:

`/content/Hindi Dub_BySHBD 240p.mp4`

If your filename is different, change `INPUT` in the later processing cell.


In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))


### Check the uploaded video

In [ ]:
INPUT='/content/Hindi Dub_BySHBD 240p.mp4'
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,duration -of default=noprint_wrappers=1 "$INPUT"


## STEP 7 — Why this notebook uses one frame at a time

We tested multi-frame processing on a Tesla T4. A 5-frame batch caused CUDA out-of-memory, while single-frame inference succeeded.

Therefore this notebook uses:

**one input frame → one RealBasicVSR inference → immediately save output**

This is slower than large batches but is the proven low-memory method for this project.

The original approach also accumulated many output tensors in RAM. This notebook avoids that by saving every frame immediately.


## STEP 8 — Create the safe one-frame inference script

In [ ]:
%%writefile /content/RealBasicVSR/inference_frames_safe.py
import os, cv2, torch, mmcv
from tqdm import tqdm
from mmcv import Config
from mmedit.models import build_model

INPUT='/content/Hindi Dub_BySHBD 240p.mp4'
OUT='/content/rbvsr_frames'
CONFIG='/content/RealBasicVSR/configs/realbasicvsr_x4.py'
CHECKPOINT='/content/RealBasicVSR/checkpoints/RealBasicVSR.pth'

os.makedirs(OUT, exist_ok=True)

cfg=Config.fromfile(CONFIG)
model=build_model(cfg.model, train_cfg=None, test_cfg=cfg.get('test_cfg'))
ck=torch.load(CHECKPOINT, map_location='cpu')
model.load_state_dict(ck['state_dict'], strict=False)
del ck
model=model.cuda().eval()

total=int(cv2.VideoCapture(INPUT).get(cv2.CAP_PROP_FRAME_COUNT))
cap=cv2.VideoCapture(INPUT)
print('Total frames:', total, flush=True)

for n in tqdm(range(total), desc='Upscaling', unit='frame'):
    ok, frame=cap.read()
    if not ok:
        print('Frame read failed:', n, flush=True)
        break

    img=torch.from_numpy(
        cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    ).float().permute(2,0,1).unsqueeze(0).unsqueeze(0).cuda()/255.0

    with torch.no_grad():
        out=model(img, test_mode=True)['output']

    result=(
        out[0,0].clamp(0,1)
        .permute(1,2,0)
        .cpu().numpy()*255
    ).astype('uint8')

    result=cv2.cvtColor(result, cv2.COLOR_RGB2BGR)

    cv2.imwrite(
        os.path.join(OUT, f'{n:05d}.png'),
        result
    )

    del img,out,result,frame
    torch.cuda.empty_cache()

cap.release()
print('DONE: frames saved to', OUT, flush=True)


## STEP 9 — Run the AI upscaling

Before a fresh run, clear incomplete frames. Then start processing. **Do not interrupt the cell while it is running.**

In [ ]:
!rm -rf /content/rbvsr_frames
!mkdir -p /content/rbvsr_frames
!/content/rbvsr_legacy/bin/python /content/RealBasicVSR/inference_frames_safe.py


## STEP 10 — Verify the generated frames

For the known test video, the expected count is 485.

In [ ]:
!ls /content/rbvsr_frames | wc -l


## STEP 11 — Create the upscaled video (GPU NOT required)

Only run this after the expected frame count is present.

The output resolution for the 422×240 test video is 1688×960.


In [ ]:
!mkdir -p /content/rbvsr_results
!ffmpeg -y -framerate 25 -i /content/rbvsr_frames/%05d.png -c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p /content/rbvsr_results/upscaled_video.mp4


## STEP 12 — Add the original audio

This copies the source AAC audio without re-encoding it.

In [ ]:
!ffmpeg -y -i /content/rbvsr_results/upscaled_video.mp4 -i "/content/Hindi Dub_BySHBD 240p.mp4" -map 0:v:0 -map 1:a:0? -c:v copy -c:a copy -shortest /content/rbvsr_results/Hindi_Dub_BySHBD_4x.mp4


## STEP 13 — Verify the final MP4

In [ ]:
!ffprobe -v error -show_entries format=duration,size -show_entries stream=index,codec_name,width,height,r_frame_rate -of default=noprint_wrappers=1 /content/rbvsr_results/Hindi_Dub_BySHBD_4x.mp4
!ls -lh /content/rbvsr_results/Hindi_Dub_BySHBD_4x.mp4


## STEP 14 — Save the final output to Google Drive

`/content` is temporary. After a successful run, copy the final MP4 to Google Drive or download it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!cp /content/rbvsr_results/Hindi_Dub_BySHBD_4x.mp4 /content/drive/MyDrive/
!ls -lh /content/drive/MyDrive/Hindi_Dub_BySHBD_4x.mp4


# Troubleshooting / Known Lessons

### NumPy error
If you see:
`AttributeError: module 'numpy' has no attribute 'bool8'`

Use:
`numpy==1.26.4`

### Wrong max sequence argument
The repository's argument is:
`--max_seq_len`

not:
`--max-seq-len`

However, this master workflow does not depend on that argument because it uses the safe one-frame script.

### GPU unavailable
If:
`nvidia-smi: command not found`

the current runtime has no GPU. Do not start AI processing.

If Colab reports:
`Cannot connect to GPU backend`

GPU access is temporarily unavailable/limited. Try another eligible runtime/account or wait.

### Colab runtime reset
Files under `/content` can disappear after a reset. The 485 frames are temporary unless copied to persistent storage.

### Heredoc error
Avoid:
`!python - <<'PY'`

Use the `%%writefile` cell provided in this notebook.

### 5-frame CUDA OOM
Do not change the safe processor to large frame batches on a T4. Single-frame inference was the proven successful method.

# Known Successful Test

Input:
- 422×240
- 25 FPS
- 19.4 seconds
- 485 frames

GPU:
- Tesla T4

Successful environment:
- Python 3.10.12
- PyTorch 1.13.1+cu117
- torchvision 0.14.1
- mmcv-full 1.7.2
- mmedit 0.16.1
- openmim 0.3.9
- NumPy 1.26.4

Successful AI run:
- 485 frames
- 4× upscale
- 1688×960
- about 3m54s
- about 2.07 frames/sec

**END OF MASTER NOTEBOOK**
